# Traffic Safety Intelligence: Accident Risk Prediction & Hotspot Analysis

An intelligent, production-ready machine learning framework for:
1. **Severity Prediction conditional on collision** (minor / major / fatal).
2. **Spatio-Temporal Hotspot Forecasting & Trend Classification** (0.2° grid, monthly panel).
3. **Time–Place Safety Twin & Expected-Harm Index** (combining spatial counts with severity distributions).
4. **Permutation & Local Ablation Explainability**.

> **Note on Data Provenance & Honest Headline Finding:**
> As documented across the platform, the bundled dataset (`indian_roads_dataset.csv`) contains synthetic/illustrative characteristics where recorded environmental features carry no learnable severity signal beyond the class prior. This notebook demonstrates the full end-to-end methodology, leak-free chronological splitting, and honest evaluation.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Ensure repo root is on python path
REPO_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_accidents, dataset_summary
from src.features import engineer_features, model_matrix_columns, severity_split_masks
from src.preprocessing import build_preprocessor
from src.train_severity import prepare_severity_data, evaluate_candidates
from src.hotspot_analysis import (
    prepare_collisions,
    build_monthly_panel,
    evaluate_count_models,
    classify_hotspot_trends,
    intensity_thresholds
)
from src.prediction import (
    load_severity_model,
    load_count_artifacts,
    predict_severity,
    expected_harm_index,
    default_input_record
)
from src.explainability import permutation_importance_table, local_contribution
from src import SEVERITY_CLASSES, SEVERITY_WEIGHTS
print('Libraries and project modules successfully loaded.')

## 1. Data Ingestion & Dataset Summary

We load the raw collision records and verify records, time span, and geographic coverage.

In [ ]:
raw_df = load_accidents()
summary = dataset_summary(raw_df)
print(f"Total collision records: {len(raw_df):,}")
print(f"Date range: {summary['date_range']['min']} -> {summary['date_range']['max']}")
print(f"Cities covered: {summary['cities']}")
print(f"Severity distribution:\n{raw_df['severity'].value_counts(normalize=True).round(3)}")
raw_df.head(3)

## 2. Feature Engineering & Spatial Binning

We apply domain-specific transformations:
- Cyclic / calendar temporal features: `hour`, `month`, `day_of_week`, `season`, `is_weekend`, `is_peak_hour`.
- Festival indicators with missing value handling.
- Geographic grid cell discretization at **0.2° (~22 km) resolution**.

In [ ]:
df_feat = engineer_features(raw_df)
print("Engineered dataset shape:", df_feat.shape)
print("Unique spatial grid cells:", df_feat['cell_id'].nunique())
df_feat[['accident_id', 'city', 'hour', 'season', 'is_peak_hour', 'is_weekend', 'cell_id', 'severity']].head()

## 3. Leakage Control & Chronological Split

Post-outcome variables (`casualties`, `risk_score`, `cause`) are strictly excluded from predictors.
Splits are strictly chronological to prevent lookahead bias:
- **Train:** 2022-01-01 to 2023-12-31
- **Validation:** 2024-01-01 to 2024-12-31
- **Held-out Test:** 2025-01-01 to 2025-04-15

In [ ]:
sev_data = prepare_severity_data()
print("Predictor features (num + cat):", sev_data['X'].columns.tolist())
print(f"Train split size: {len(sev_data['splits']['train']):,} rows")
print(f"Val split size:   {len(sev_data['splits']['val']):,} rows")
print(f"Test split size:  {len(sev_data['splits']['test']):,} rows")

## 4. Severity Model Training & Baseline Benchmarking

We benchmark multiple classifiers against two probability baselines:
- `prior`: Predicts training class frequencies (macro-F1 & log-loss floor).
- `dummy`: Stratified random guessing.
- `logreg`: Logistic Regression with class-weight balance.
- `rf`: Random Forest with balanced sub-sampling.
- `hgb`: Histogram Gradient Boosting.

In [ ]:
val_comp, fitted = evaluate_candidates(sev_data['X'], sev_data['y'], sev_data['splits'])
val_comp

### Test Evaluation of Winning Pipeline
We test the pipeline on the held-out 2025 data.

In [ ]:
from src.evaluation import classification_metrics, probability_metrics, confusion_df

pipe = load_severity_model()
test_idx = sev_data['splits']['test']
X_test = sev_data['X'].loc[test_idx]
y_test = sev_data['y'].loc[test_idx]

probs_test = pipe.predict_proba(X_test)
preds_test = pipe.predict(X_test)

metrics = {
    **classification_metrics(y_test, preds_test, SEVERITY_CLASSES),
    **probability_metrics(y_test, probs_test, SEVERITY_CLASSES)
}
for k, v in metrics.items():
    print(f"{k:20s}: {v:.4f}")

print("\nConfusion Matrix (Test):")
print(confusion_df(y_test, preds_test, SEVERITY_CLASSES))

## 5. Hotspot Panel & Monthly Collision Count Forecasting

Collisions are aggregated into a monthly spatio-temporal panel `(cell_id, month)`.
Cells with sufficient historical exposure (≥12 collisions over panel) are retained.
We benchmark count forecast models on held-out months (2024-10 to 2025-03):
- `cell_mean`: Cell historical average
- `seasonal_avg`: Average for the same month across past years
- `poisson`: Poisson generalized linear model with lag features
- `prev_month`: Naive lag-1 baseline

In [ ]:
collisions = prepare_collisions(raw_df)
panel, eligible_cells = build_monthly_panel(collisions)
print(f"Eligible grid cells: {len(eligible_cells)}")
print(f"Panel shape: {panel.shape} (cell-months)")

count_comp, fitted_count_models = evaluate_count_models(panel)
count_comp

## 6. Emerging Hotspots Early Warning System

Using deterministic rule-based early warning:
- **Persistent hotspot:** ≥50% of the last 3 months in the High intensity tier.
- **Emerging hotspot:** Recent 3-month mean ≥ 1.5× prior 12-month baseline.
- **Declining:** Recent 3-month mean ≤ 0.7× prior baseline.
- **Stable:** Within normal baseline fluctuations.
- **Insufficient evidence:** Sparse baseline (<5 collisions).

In [ ]:
trends = classify_hotspot_trends(panel, intensity_thresholds(panel))
print("Trend Distribution:")
print(trends['trend_label'].value_counts())

trends[['cell_id', 'latitude', 'longitude', 'baseline_collisions', 'recent_collisions', 'trend_ratio', 'trend_label']].head(10)

## 7. Model Explainability: Global Permutation & Local Ablation

Understanding model behavior transparently without confounding causality.

In [ ]:
val_idx = sev_data['splits']['val']
X_val = sev_data['X'].loc[val_idx]
y_val = sev_data['y'].loc[val_idx]

imp = permutation_importance_table(pipe, X_val, y_val, n_repeats=3)
imp.head(10)

In [ ]:
# Local ablation on a sample record
rec = default_input_record()
X_row = pd.DataFrame([rec])[sev_data['X'].columns]
bg = sev_data['X'].iloc[:300]
contrib = local_contribution(pipe, X_row, bg)
contrib.head(8)

## 8. Time–Place Safety Twin & Expected-Harm Index

Combining forecasted accident volume with the conditional severity distribution into a single actionable index:
$$\text{Expected Harm} = \widehat{\text{Count}} \times \sum_{c} P(\text{Severity}=c) \times w_c$$

In [ ]:
sample_cell = eligible_cells[0]
sample_forecast = 3.5  # forecasted collisions next month

pred = predict_severity(rec)
prob_dict = dict(zip(pred['classes'], pred['probabilities']))

harm_standard = expected_harm_index(sample_forecast, prob_dict, SEVERITY_WEIGHTS)

# Custom safety policy weights emphasizing fatal prevention
custom_weights = {"minor": 1.0, "major": 5.0, "fatal": 25.0}
harm_custom = expected_harm_index(sample_forecast, prob_dict, custom_weights)

print(f"Cell: {sample_cell}")
print(f"Forecasted collisions: {sample_forecast}")
print("Conditional severity probabilities:", {k: round(v, 3) for k, v in prob_dict.items()})
print(f"Standard Expected Harm Index: {harm_standard:.2f}")
print(f"Policy-Adjusted Harm Index:   {harm_custom:.2f}")

## 9. Synthesis & Practical Takeaways

1. **System Integrity**: All components (data loading, preprocessing, feature engineering, model training, evaluation, explainability, safety twin) are modular, leak-free, and fully reproducible.
2. **Methodology Validation**: The architecture functions as intended. In real-world deployments with actual road telematics or official STATS19 data, the pipeline will capture genuine spatial risk and severity patterns without modifying the code.